In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlambda/2)**2
    return 2 * R * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

In [ ]:
df = pd.read_csv("data/fraudTrain.csv")

In [ ]:
df['distance_km'] = haversine(df['lat'], df['long'], df['merch_lat'], df['merch_long'])
df['dob'] = pd.to_datetime(df['dob'])
df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])
df['age'] = (df['trans_date_trans_time'] - df['dob']).dt.days // 365
df['hour'] = df['trans_date_trans_time'].dt.hour

In [ ]:
features = ['category', 'amt', 'gender', 'age', 'distance_km', 'hour']
X = df[features].copy()
y = df['is_fraud']
X['gender'] = X['gender'].map({'M': 0, 'F': 1})
le_cat = LabelEncoder()
X['category'] = le_cat.fit_transform(X['category'])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_split=20,
    min_samples_leaf=10,
    max_features='sqrt',
    class_weight='balanced_subsample',
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print("Train accuracy:", model.score(X_train, y_train))
print("Test accuracy:", model.score(X_test, y_test))

In [ ]:
joblib.dump(model, 'model/fraud_model.pkl')
joblib.dump(le_cat, 'model/category_encoder.pkl')
joblib.dump(features, 'model/features.pkl')